In [43]:
import pandas as pd
import re
import numpy as np
import scipy.stats as st

In [28]:
# Groud Truths del escenario 1
def ground_truths_escenario1(log_text):
    # Convertir a mayúsculas para evitar problemas de case sensitive
    log = str(log_text).upper()
    
    # Regla 1: Acceso normal de alumno (uid=1000)
    if 'TYPE=SYSCALL' in log and any(cmd in log for cmd in ['COMM="CAT"', 'COMM="TAIL"', 'COMM="GREP"']) and 'UID=1000' in log:
        return ['BAJO', 'INFO', 'INFORMACIÓN']
    # Regla 2: Ejecución de sudo por root (euid=0)
    elif 'TYPE=SYSCALL' in log and 'COMM="SUDO"' in log and 'EUID=0' in log:
        return ['MEDIO']
    # Regla 3: Eventos auxiliares que no tienen riesgo directo
    elif 'TYPE=CWD' in log or 'TYPE=PROCTITLE' in log:
        return ['INFO', 'INFORMACIÓN']
    # Por defecto, cualquier otro evento de soporte del sistema es ruido/info
    return ['INFORMACIÓN', 'INFO', 'BAJO']

# Groud Truths del escenario 2
def ground_truths_escenario2(log_text):
    log = str(log_text).upper()
    
    # Regla 1 estricta: cat /etc/shadow en HEX DEBE ser ALTO
    if '636174202F6574632F736861646F77' in log:
        return ['ALTO']
    # Regla 2: Otros intentos de escalada o auth fallida
    elif 'RES=FAILED' in log and ('TYPE=USER_CMD' in log or 'TYPE=USER_AUTH' in log):
        return ['ALTO', 'MEDIO']
    # Manejo de errores de parseo del LLM
    elif 'ERROR_PARSE' in log or 'FALLO_LLM' in log:
        return ['ERROR']
    # Por defecto
    return ['INFO', 'BAJO', 'INFORMACIÓN']

# Groud Truths del escenario 3
def ground_truths_escenario3(log_text):
    log = str(log_text).upper()
    
    if 'TYPE=SYSCALL' in log and ('AUID="ROOT"' in log or 'UID="ROOT"' in log) and any(cmd in log for cmd in ['COMM="APT-GET"', 'COMM="SYSTEMCTL"']):
        return ['INFO', 'BAJO', 'INFORMACIÓN']
    elif 'TYPE=SYSCALL' in log and 'COMM="USERADD"' in log:
        return ['MEDIO']
    elif 'ERROR_PARSE' in log or 'FALLO_LLM' in log:
        return ['ERROR']
    else:
        return ['INFO', 'BAJO', 'INFORMACIÓN']

# Groud Truths del escenario 4
def ground_truths_escenario4(log_text):
    log = str(log_text).upper()
    
    if 'TYPE=SYSCALL' in log and any(cmd in log for cmd in ['COMM="VISUDO"', 'COMM="TOUCH"', 'COMM="CHMOD"']):
        return ['ALTO', 'MEDIO']
    elif 'TYPE=SYSCALL' in log and 'COMM="SUDO"' in log:
        return ['MEDIO']
    elif 'TYPE=PATH' in log and ('NAMETYPE=DELETE' in log or 'NAMETYPE=CREATE' in log):
        return ['ALTO', 'MEDIO']
    elif 'ERROR_PARSE' in log or 'FALLO_LLM' in log:
        return ['ERROR']
    else:
        return ['INFO', 'BAJO', 'INFORMACIÓN']

# Groud Truths del escenario 5
def ground_truths_escenario5(log_text):
    log = str(log_text).upper()
    
    # Extraer el AUID numérico para no depender de nombres en el código
    match_auid = re.search(r'AUID="?(\d+)"?', log)
    auid_num = int(match_auid.group(1)) if match_auid else -1

    if 'TYPE=SYSCALL' in log and 'EXIT=-13' in log and any(cmd in log for cmd in ['COMM="CAT"', 'COMM="HEAD"', 'COMM="CP"']):
        return ['ALTO']
    elif 'TYPE=SYSCALL' in log and auid_num > 1000 and any(cmd in log for cmd in ['COMM="SUDO"', 'COMM="SU"']):
        return ['ALTO', 'MEDIO']
    elif 'TYPE=SYSCALL' in log and any(cmd in log for cmd in ['COMM="SSHD"', 'COMM="LOGIN"', 'COMM="(SYSTEMD)"', 'COMM="CRON"']):
        return ['INFO', 'BAJO', 'INFORMACION', 'INFORMADO']
    elif 'TYPE=SYSCALL' in log and auid_num == 1000 and 'COMM="SUDO"' in log:
        return ['INFO', 'BAJO', 'INFORMACION', 'INFORMADO']
    else:
        return ['INFO', 'BAJO', 'INFORMACION', 'INFORMADO']

In [29]:
# Función para clacular la precisión
def evaluar_por_orden(path_completo, path_evaluar, ground_truths):
    """
    Evalúa la precisión comparando fila por fila y retorna solo la precisión del LLM.
    """
    df_completo = pd.read_csv(path_completo)
    df_evaluar = pd.read_csv(path_evaluar)
    
    if len(df_completo) != len(df_evaluar):
        # Retorna None si hay un error en los archivos para que no rompa el loop
        return None 
        
    verdad_en_orden = df_completo['Log'].apply(ground_truths).tolist()
    
    aciertos_llm = 0
    total_llm_evaluado = 0
    total_filas = len(df_evaluar)
    
    for i in range(total_filas):
        etiquetas_reales = verdad_en_orden[i]
        
        justificacion = str(df_evaluar.loc[i, 'Justificacion'] if 'Justificacion' in df_evaluar.columns else '').strip()
        es_omitido = 'Omitido' in justificacion
        
        if not es_omitido:
            prediccion_llm = str(df_evaluar.loc[i, 'Riesgo']).upper().strip()
            total_llm_evaluado += 1
            
            if prediccion_llm in etiquetas_reales:
                aciertos_llm += 1
                
    # Cálculo final
    precision_llm = round((aciertos_llm / total_llm_evaluado) * 100,2) if total_llm_evaluado > 0 else 0
    
    return precision_llm

In [ ]:
# ESCENARIO 1
E1_PHI_R1 = '../results/prompt1/escenario1_resultados_raw_completo_phi3mini.csv'
E1_PHI_R2 = '../results/prompt1/escenario1_resultados_raw_completo_phi3mini_2.csv'
E1_PHI_R3 = '../results/prompt1/escenario1_resultados_raw_completo_phi3mini_3.csv'
E1_PHI_R4 = '../results/prompt1/escenario1_resultados_raw_completo_phi3mini_4.csv'
E1_PHI_R5 = '../results/prompt1/escenario1_resultados_raw_completo_phi3mini_5.csv'

E1_LLAMA_R1 = '../results/prompt1/escenario1_resultados_raw_completo_llama3-2_3b.csv'
E1_LLAMA_R2 = '../results/prompt1/escenario1_resultados_raw_completo_llama3-2_3b_2.csv'
E1_LLAMA_R3 = '../results/prompt1/escenario1_resultados_raw_completo_llama3-2_3b_3.csv'
E1_LLAMA_R4 = '../results/prompt1/escenario1_resultados_raw_completo_llama3-2_3b_4.csv'
E1_LLAMA_R5 = '../results/prompt1/escenario1_resultados_raw_completo_llama3-2_3b_5.csv'

ruta_verdad = '../results/prompt1/escenario1_resultados_raw_completo_phi3mini.csv'
# Precisión
e1_phi_1 = evaluar_por_orden(ruta_verdad, E1_PHI_R1, ground_truths_escenario1)
e1_phi_2 = evaluar_por_orden(ruta_verdad, E1_PHI_R2, ground_truths_escenario1)
e1_phi_3 = evaluar_por_orden(ruta_verdad, E1_PHI_R3, ground_truths_escenario1)
e1_phi_4 = evaluar_por_orden(ruta_verdad, E1_PHI_R4, ground_truths_escenario1)
e1_phi_5 = evaluar_por_orden(ruta_verdad, E1_PHI_R5, ground_truths_escenario1)

precisiones_e1_phi = [e1_phi_1, e1_phi_2, e1_phi_3, e1_phi_4, e1_phi_5]
promedio_e1_phi = round(np.mean(precisiones_e1_phi), 2)

e1_llama_1 = evaluar_por_orden(ruta_verdad, E1_LLAMA_R1, ground_truths_escenario1)
e1_llama_2 = evaluar_por_orden(ruta_verdad, E1_LLAMA_R2, ground_truths_escenario1)
e1_llama_3 = evaluar_por_orden(ruta_verdad, E1_LLAMA_R3, ground_truths_escenario1)
e1_llama_4 = evaluar_por_orden(ruta_verdad, E1_LLAMA_R4, ground_truths_escenario1)
e1_llama_5 = evaluar_por_orden(ruta_verdad, E1_LLAMA_R5, ground_truths_escenario1)

precisiones_e1_llama = [e1_llama_1, e1_llama_2, e1_llama_3, e1_llama_4, e1_llama_5]
promedio_e1_llama = round(np.mean(precisiones_e1_llama), 2)

# Desvío estandar
e1_desvio_estandar_phi = round(np.std(precisiones_e1_phi, ddof=1), 2)

# Intervalo de Confianza del 95%
e1_grados_libertad = len(precisiones_e1_phi) - 1
e1_error_estandar = st.sem(precisiones_e1_phi)
e1_intervalo_confianza = st.t.interval(0.95, e1_grados_libertad, loc=promedio_e1_phi, scale=e1_error_estandar)

print("RESULTADOS ESCENARIO 1")
print("="*64)
print("phi3:mini")
print(f"Precisión Repetición 1: {e1_phi_1}%")
print(f"Precisión Repetición 2: {e1_phi_2}%")
print(f"Precisión Repetición 3: {e1_phi_3}%")
print(f"Precisión Repetición 4: {e1_phi_4}%")
print(f"Precisión Repetición 5: {e1_phi_5}%")
print(f"Promedio: {promedio_e1_phi}%")
print(f"Desvío estandar: {e1_desvio_estandar_phi}%")
print(f"Intervalo de confianza del 95%: [{round(e1_intervalo_confianza[0], 2)}, {round(e1_intervalo_confianza[1], 2)}]%")
print("-"*65)
print("llama3.2:3b")
print(f"Precisión Repetición 1: {e1_llama_1}%")
print(f"Precisión Repetición 2: {e1_llama_2}%")
print(f"Precisión Repetición 3: {e1_llama_3}%")
print(f"Precisión Repetición 4: {e1_llama_4}%")
print(f"Precisión Repetición 5: {e1_llama_5}%")
print(f"Promedio: {promedio_e1_llama}%")

RESULTADOS ESCENARIO 1
phi3:mini
Precisión Repetición 1: 73.38%
Precisión Repetición 2: 60.43%
Precisión Repetición 3: 63.31%
Precisión Repetición 4: 61.87%
Precisión Repetición 5: 62.59%
Promedio: 64.32%
Desvío estandar: 5.18%
Intervalo de confianza del 95%: [57.89, 70.75]%
-----------------------------------------------------------------
llama3.2:3b
Precisión Repetición 1: 53.96%
Precisión Repetición 2: 54.68%
Precisión Repetición 3: 49.64%
Precisión Repetición 4: 51.08%
Precisión Repetición 5: 51.8%
Promedio: 52.23%


In [50]:
# Precisión de las repeticiones del escenario 2
E2_PHI_R1 = '../results/prompt1/escenario2_resultados_raw_completo_phi3mini.csv'
E2_PHI_R2 = '../results/prompt1/escenario2_resultados_raw_completo_phi3mini_2.csv'
E2_PHI_R3 = '../results/prompt1/escenario2_resultados_raw_completo_phi3mini_3.csv'
E2_PHI_R4 = '../results/prompt1/escenario2_resultados_raw_completo_phi3mini_4.csv'
E2_PHI_R5 = '../results/prompt1/escenario2_resultados_raw_completo_phi3mini_5.csv'

E2_LLAMA_R1 = '../results/prompt1/escenario2_resultados_raw_completo_llama3-2_3b.csv'
E2_LLAMA_R2 = '../results/prompt1/escenario2_resultados_raw_completo_llama3-2_3b_2.csv'
E2_LLAMA_R3 = '../results/prompt1/escenario2_resultados_raw_completo_llama3-2_3b_3.csv'
E2_LLAMA_R4 = '../results/prompt1/escenario2_resultados_raw_completo_llama3-2_3b_4.csv'
E2_LLAMA_R5 = '../results/prompt1/escenario2_resultados_raw_completo_llama3-2_3b_5.csv'

ruta_verdad = '../results/prompt1/escenario2_resultados_raw_completo_phi3mini.csv'

e2_phi_1 = evaluar_por_orden(ruta_verdad, E2_PHI_R1, ground_truths_escenario2)
e2_phi_2 = evaluar_por_orden(ruta_verdad, E2_PHI_R2, ground_truths_escenario2)
e2_phi_3 = evaluar_por_orden(ruta_verdad, E2_PHI_R3, ground_truths_escenario2)
e2_phi_4 = evaluar_por_orden(ruta_verdad, E2_PHI_R4, ground_truths_escenario2)
e2_phi_5 = evaluar_por_orden(ruta_verdad, E2_PHI_R5, ground_truths_escenario2)

precisiones_e2_phi = [e2_phi_1, e2_phi_2, e2_phi_3, e2_phi_4, e2_phi_5]
promedio_e2_phi = round(np.mean(precisiones_e2_phi), 2)

e2_llama_1 = evaluar_por_orden(ruta_verdad, E2_LLAMA_R1, ground_truths_escenario2)
e2_llama_2 = evaluar_por_orden(ruta_verdad, E2_LLAMA_R2, ground_truths_escenario2)
e2_llama_3 = evaluar_por_orden(ruta_verdad, E2_LLAMA_R3, ground_truths_escenario2)
e2_llama_4 = evaluar_por_orden(ruta_verdad, E2_LLAMA_R4, ground_truths_escenario2)
e2_llama_5 = evaluar_por_orden(ruta_verdad, E2_LLAMA_R5, ground_truths_escenario2)

precisiones_e2_llama = [e2_llama_1, e2_llama_2, e2_llama_3, e2_llama_4, e2_llama_5]
promedio_e2_llama = round(np.mean(precisiones_e2_llama), 2)

# Desvío estandar
e2_desvio_estandar_phi = round(np.std(precisiones_e2_phi, ddof=1), 2)

# Intervalo de Confianza del 95%
if e2_desvio_estandar_phi == 0.0:
    e2_intervalo_texto = "No calculable (Varianza nula)"
else:
    e2_grados_libertad = len(precisiones_e2_phi) - 1
    e2_error_estandar = st.sem(precisiones_e2_phi)
    e2_intervalo_confianza = st.t.interval(0.95, e2_grados_libertad, loc=promedio_e2_phi, scale=e2_error_estandar)

    e2_limite_inferior = max(0.0, e2_intervalo_confianza[0])
    e2_limite_superior = min(100.0, e2_intervalo_confianza[1])
    e2_intervalo_texto = f"[{e2_limite_inferior}%, {e2_limite_superior}%]"

print("RESULTADOS ESCENARIO 2")
print("="*64)
print("phi3:mini")
print(f"Precisión Repetición 1: {e2_phi_1}%")
print(f"Precisión Repetición 2: {e2_phi_2}%")
print(f"Precisión Repetición 3: {e2_phi_3}%")
print(f"Precisión Repetición 4: {e2_phi_4}%")
print(f"Precisión Repetición 5: {e2_phi_5}%")
print(f"Promedio: {promedio_e2_phi}%")
print(f"Desvío estandar: {e2_desvio_estandar_phi}%")
print(f"Intervalo de confianza del 95%: {e2_intervalo_texto}")
print("-"*65)
print("llama3.2:3b")
print(f"Precisión Repetición 1: {e2_llama_1}%")
print(f"Precisión Repetición 2: {e2_llama_2}%")
print(f"Precisión Repetición 3: {e2_llama_3}%")
print(f"Precisión Repetición 4: {e2_llama_4}%")
print(f"Precisión Repetición 5: {e2_llama_5}%")
print(f"Promedio: {promedio_e2_llama}%")

RESULTADOS ESCENARIO 2
phi3:mini
Precisión Repetición 1: 85.71%
Precisión Repetición 2: 85.71%
Precisión Repetición 3: 85.71%
Precisión Repetición 4: 85.71%
Precisión Repetición 5: 85.71%
Promedio: 85.71%
Desvío estandar: 0.0%
Intervalo de confianza del 95%: No calculable (Varianza nula)
-----------------------------------------------------------------
llama3.2:3b
Precisión Repetición 1: 28.57%
Precisión Repetición 2: 57.14%
Precisión Repetición 3: 28.57%
Precisión Repetición 4: 42.86%
Precisión Repetición 5: 14.29%
Promedio: 34.29%


In [48]:
# Precisión de las repeticiones del escenario 3
E3_PHI_R1 = '../results/prompt1/escenario3_resultados_raw_completo_phi3mini.csv'
E3_PHI_R2 = '../results/prompt1/escenario3_resultados_raw_completo_phi3mini_2.csv'
E3_PHI_R3 = '../results/prompt1/escenario3_resultados_raw_completo_phi3mini_3.csv'
E3_PHI_R4 = '../results/prompt1/escenario3_resultados_raw_completo_phi3mini_4.csv'
E3_PHI_R5 = '../results/prompt1/escenario3_resultados_raw_completo_phi3mini_5.csv'

E3_LLAMA_R1 = '../results/prompt1/escenario3_resultados_raw_completo_llama3-2_3b.csv'
E3_LLAMA_R2 = '../results/prompt1/escenario3_resultados_raw_completo_llama3-2_3b_2.csv'
E3_LLAMA_R3 = '../results/prompt1/escenario3_resultados_raw_completo_llama3-2_3b_3.csv'
E3_LLAMA_R4 = '../results/prompt1/escenario3_resultados_raw_completo_llama3-2_3b_4.csv'
E3_LLAMA_R5 = '../results/prompt1/escenario3_resultados_raw_completo_llama3-2_3b_5.csv'

ruta_verdad = '../results/prompt1/escenario3_resultados_raw_completo_phi3mini.csv'

e3_phi_1 = evaluar_por_orden(ruta_verdad, E3_PHI_R1, ground_truths_escenario3)
e3_phi_2 = evaluar_por_orden(ruta_verdad, E3_PHI_R2, ground_truths_escenario3)
e3_phi_3 = evaluar_por_orden(ruta_verdad, E3_PHI_R3, ground_truths_escenario3)
e3_phi_4 = evaluar_por_orden(ruta_verdad, E3_PHI_R4, ground_truths_escenario3)
e3_phi_5 = evaluar_por_orden(ruta_verdad, E3_PHI_R5, ground_truths_escenario3)

precisiones_e3_phi = [e3_phi_1, e3_phi_2, e3_phi_3, e3_phi_4, e3_phi_5]
promedio_e3_phi = round(np.mean(precisiones_e3_phi), 2)

e3_llama_1 = evaluar_por_orden(ruta_verdad, E3_LLAMA_R1, ground_truths_escenario3)
e3_llama_2 = evaluar_por_orden(ruta_verdad, E3_LLAMA_R2, ground_truths_escenario3)
e3_llama_3 = evaluar_por_orden(ruta_verdad, E3_LLAMA_R3, ground_truths_escenario3)
e3_llama_4 = evaluar_por_orden(ruta_verdad, E3_LLAMA_R4, ground_truths_escenario3)
e3_llama_5 = evaluar_por_orden(ruta_verdad, E3_LLAMA_R5, ground_truths_escenario3)

precisiones_e3_llama = [e3_llama_1, e3_llama_2, e3_llama_3, e3_llama_4, e3_llama_5]
promedio_e3_llama = round(np.mean(precisiones_e3_llama), 2)

# Desvío estandar
e3_desvio_estandar_phi = round(np.std(precisiones_e3_phi, ddof=1), 2)

# Intervalo de Confianza del 95%
e3_grados_libertad = len(precisiones_e3_phi) - 1
e3_error_estandar = st.sem(precisiones_e3_phi)
e3_intervalo_confianza = st.t.interval(0.95, e3_grados_libertad, loc=promedio_e3_phi, scale=e3_error_estandar)
e3_limite_inferior = max(0.0, e3_intervalo_confianza[0])
e3_limite_superior = min(100.0, e3_intervalo_confianza[1])

print("RESULTADOS ESCENARIO 3")
print("="*64)
print("phi3:mini")
print(f"Precisión Repetición 1: {e3_phi_1}%")
print(f"Precisión Repetición 2: {e3_phi_2}%")
print(f"Precisión Repetición 3: {e3_phi_3}%")
print(f"Precisión Repetición 4: {e3_phi_4}%")
print(f"Precisión Repetición 5: {e3_phi_5}%")
print(f"Promedio: {promedio_e3_phi}%")
print(f"Desvío estandar: {e3_desvio_estandar_phi}%")
print(f"Intervalo de confianza del 95%: [{round(e3_limite_inferior, 2)}, {round(e3_limite_superior, 2)}]%")
print("-"*65)
print("llama3.2:3b")
print(f"Precisión Repetición 1: {e3_llama_1}%")
print(f"Precisión Repetición 2: {e3_llama_2}%")
print(f"Precisión Repetición 3: {e3_llama_3}%")
print(f"Precisión Repetición 4: {e3_llama_4}%")
print(f"Precisión Repetición 5: {e3_llama_5}%")
print(f"Promedio: {promedio_e3_llama}%")

RESULTADOS ESCENARIO 3
phi3:mini
Precisión Repetición 1: 10.0%
Precisión Repetición 2: 0.0%
Precisión Repetición 3: 20.0%
Precisión Repetición 4: 10.0%
Precisión Repetición 5: 0.0%
Promedio: 8.0%
Desvío estandar: 8.37%
Intervalo de confianza del 95%: [0.0, 18.39]%
-----------------------------------------------------------------
llama3.2:3b
Precisión Repetición 1: 20.0%
Precisión Repetición 2: 30.0%
Precisión Repetición 3: 20.0%
Precisión Repetición 4: 30.0%
Precisión Repetición 5: 20.0%
Promedio: 24.0%


In [46]:
# Precisión de las repeticiones del escenario 4
E4_PHI_R1 = '../results/prompt1/escenario4_resultados_raw_completo_phi3mini.csv'
E4_PHI_R2 = '../results/prompt1/escenario4_resultados_raw_completo_phi3mini_2.csv'
E4_PHI_R3 = '../results/prompt1/escenario4_resultados_raw_completo_phi3mini_3.csv'
E4_PHI_R4 = '../results/prompt1/escenario4_resultados_raw_completo_phi3mini_4.csv'
E4_PHI_R5 = '../results/prompt1/escenario4_resultados_raw_completo_phi3mini_5.csv'

E4_LLAMA_R1 = '../results/prompt1/escenario4_resultados_raw_completo_llama3-2_3b.csv'
E4_LLAMA_R2 = '../results/prompt1/escenario4_resultados_raw_completo_llama3-2_3b_2.csv'
E4_LLAMA_R3 = '../results/prompt1/escenario4_resultados_raw_completo_llama3-2_3b_3.csv'
E4_LLAMA_R4 = '../results/prompt1/escenario4_resultados_raw_completo_llama3-2_3b_4.csv'
E4_LLAMA_R5 = '../results/prompt1/escenario4_resultados_raw_completo_llama3-2_3b_5.csv'

ruta_verdad = '../results/prompt1/escenario4_resultados_raw_completo_phi3mini.csv'

e4_phi_1 = evaluar_por_orden(ruta_verdad, E4_PHI_R1, ground_truths_escenario4)
e4_phi_2 = evaluar_por_orden(ruta_verdad, E4_PHI_R2, ground_truths_escenario4)
e4_phi_3 = evaluar_por_orden(ruta_verdad, E4_PHI_R3, ground_truths_escenario4)
e4_phi_4 = evaluar_por_orden(ruta_verdad, E4_PHI_R4, ground_truths_escenario4)
e4_phi_5 = evaluar_por_orden(ruta_verdad, E4_PHI_R5, ground_truths_escenario4)

precisiones_e4_phi = [e4_phi_1, e4_phi_2, e4_phi_3, e4_phi_4, e4_phi_5]
promedio_e4_phi = round(np.mean(precisiones_e4_phi), 2)

e4_llama_1 = evaluar_por_orden(ruta_verdad, E4_LLAMA_R1, ground_truths_escenario4)
e4_llama_2 = evaluar_por_orden(ruta_verdad, E4_LLAMA_R2, ground_truths_escenario4)
e4_llama_3 = evaluar_por_orden(ruta_verdad, E4_LLAMA_R3, ground_truths_escenario4)
e4_llama_4 = evaluar_por_orden(ruta_verdad, E4_LLAMA_R4, ground_truths_escenario4)
e4_llama_5 = evaluar_por_orden(ruta_verdad, E4_LLAMA_R5, ground_truths_escenario4)

precisiones_e4_llama = [e4_llama_1, e4_llama_2, e4_llama_3, e4_llama_4, e4_llama_5]
promedio_e4_llama = round(np.mean(precisiones_e4_llama), 2)

# Desvío estandar
e4_desvio_estandar_phi = round(np.std(precisiones_e4_phi, ddof=1), 2)

# Intervalo de Confianza del 95%
e4_grados_libertad = len(precisiones_e4_phi) - 1
e4_error_estandar = st.sem(precisiones_e4_phi)
e4_intervalo_confianza = st.t.interval(0.95, e4_grados_libertad, loc=promedio_e4_phi, scale=e4_error_estandar)

print("RESULTADOS ESCENARIO 4")
print("="*64)
print("phi3:mini")
print(f"Precisión Repetición 1: {e4_phi_1}%")
print(f"Precisión Repetición 2: {e4_phi_2}%")
print(f"Precisión Repetición 3: {e4_phi_3}%")
print(f"Precisión Repetición 4: {e4_phi_4}%")
print(f"Precisión Repetición 5: {e4_phi_5}%")
print(f"Promedio: {promedio_e4_phi}%")
print(f"Desvío estandar: {e4_desvio_estandar_phi}%")
print(f"Intervalo de confianza del 95%: [{round(e4_intervalo_confianza[0], 2)}, {round(e4_intervalo_confianza[1], 2)}]%")
print("-"*65)
print("llama3.2:3b")
print(f"Precisión Repetición 1: {e4_llama_1}%")
print(f"Precisión Repetición 2: {e4_llama_2}%")
print(f"Precisión Repetición 3: {e4_llama_3}%")
print(f"Precisión Repetición 4: {e4_llama_4}%")
print(f"Precisión Repetición 5: {e4_llama_5}%")
print(f"Promedio: {promedio_e4_llama}%")

RESULTADOS ESCENARIO 4
phi3:mini
Precisión Repetición 1: 73.68%
Precisión Repetición 2: 75.0%
Precisión Repetición 3: 71.05%
Precisión Repetición 4: 72.37%
Precisión Repetición 5: 68.42%
Promedio: 72.1%
Desvío estandar: 2.53%
Intervalo de confianza del 95%: [68.96, 75.24]%
-----------------------------------------------------------------
llama3.2:3b
Precisión Repetición 1: 68.42%
Precisión Repetición 2: 72.37%
Precisión Repetición 3: 65.79%
Precisión Repetición 4: 64.47%
Precisión Repetición 5: 61.84%
Promedio: 66.58%


In [47]:
# Precisión de las repeticiones del escenario 5
E5_PHI_R1 = '../results/prompt1/escenario5_resultados_raw_completo_phi3mini.csv'
E5_PHI_R2 = '../results/prompt1/escenario5_resultados_raw_completo_phi3mini_2.csv'
E5_PHI_R3 = '../results/prompt1/escenario5_resultados_raw_completo_phi3mini_3.csv'
E5_PHI_R4 = '../results/prompt1/escenario5_resultados_raw_completo_phi3mini_4.csv'
E5_PHI_R5 = '../results/prompt1/escenario5_resultados_raw_completo_phi3mini_5.csv'

E5_LLAMA_R1 = '../results/prompt1/escenario5_resultados_raw_completo_llama3-2_3b.csv'
E5_LLAMA_R2 = '../results/prompt1/escenario5_resultados_raw_completo_llama3-2_3b_2.csv'
E5_LLAMA_R3 = '../results/prompt1/escenario5_resultados_raw_completo_llama3-2_3b_3.csv'
E5_LLAMA_R4 = '../results/prompt1/escenario5_resultados_raw_completo_llama3-2_3b_4.csv'
E5_LLAMA_R5 = '../results/prompt1/escenario5_resultados_raw_completo_llama3-2_3b_5.csv'

ruta_verdad = '../results/prompt1/escenario5_resultados_raw_completo_phi3mini.csv'

e5_phi_1 = evaluar_por_orden(ruta_verdad, E5_PHI_R1, ground_truths_escenario5)
e5_phi_2 = evaluar_por_orden(ruta_verdad, E5_PHI_R2, ground_truths_escenario5)
e5_phi_3 = evaluar_por_orden(ruta_verdad, E5_PHI_R3, ground_truths_escenario5)
e5_phi_4 = evaluar_por_orden(ruta_verdad, E5_PHI_R4, ground_truths_escenario5)
e5_phi_5 = evaluar_por_orden(ruta_verdad, E5_PHI_R5, ground_truths_escenario5)

precisiones_e5_phi = [e5_phi_1, e5_phi_2, e5_phi_3, e5_phi_4, e5_phi_5]
promedio_e5_phi = round(np.mean(precisiones_e5_phi), 2)

e5_llama_1 = evaluar_por_orden(ruta_verdad, E5_LLAMA_R1, ground_truths_escenario5)
e5_llama_2 = evaluar_por_orden(ruta_verdad, E5_LLAMA_R2, ground_truths_escenario5)
e5_llama_3 = evaluar_por_orden(ruta_verdad, E5_LLAMA_R3, ground_truths_escenario5)
e5_llama_4 = evaluar_por_orden(ruta_verdad, E5_LLAMA_R4, ground_truths_escenario5)
e5_llama_5 = evaluar_por_orden(ruta_verdad, E5_LLAMA_R5, ground_truths_escenario5)

precisiones_e5_llama = [e5_llama_1, e5_llama_2, e5_llama_3, e5_llama_4, e5_llama_5]
promedio_e5_llama = round(np.mean(precisiones_e5_llama), 2)

# Desvío estandar
e5_desvio_estandar_phi = round(np.std(precisiones_e5_phi, ddof=1), 2)

# Intervalo de Confianza del 95%
e5_grados_libertad = len(precisiones_e5_phi) - 1
e5_error_estandar = st.sem(precisiones_e5_phi)
e5_intervalo_confianza = st.t.interval(0.95, e5_grados_libertad, loc=promedio_e5_phi, scale=e5_error_estandar)

print("RESULTADOS ESCENARIO 5")
print("="*64)
print("phi3:mini")
print(f"Precisión Repetición 1: {e5_phi_1}%")
print(f"Precisión Repetición 2: {e5_phi_2}%")
print(f"Precisión Repetición 3: {e5_phi_3}%")
print(f"Precisión Repetición 4: {e5_phi_4}%")
print(f"Precisión Repetición 5: {e5_phi_5}%")
print(f"Promedio: {promedio_e5_phi}%")
print(f"Desvío estandar: {e5_desvio_estandar_phi}%")
print(f"Intervalo de confianza del 95%: [{round(e5_intervalo_confianza[0], 2)}, {round(e5_intervalo_confianza[1], 2)}]%")
print("-"*65)
print("llama3.2:3b")
print(f"Precisión Repetición 1: {e5_llama_1}%")
print(f"Precisión Repetición 2: {e5_llama_2}%")
print(f"Precisión Repetición 3: {e5_llama_3}%")
print(f"Precisión Repetición 4: {e5_llama_4}%")
print(f"Precisión Repetición 5: {e5_llama_5}%")
print(f"Promedio: {promedio_e5_llama}%")

RESULTADOS ESCENARIO 5
phi3:mini
Precisión Repetición 1: 73.08%
Precisión Repetición 2: 72.65%
Precisión Repetición 3: 70.09%
Precisión Repetición 4: 69.66%
Precisión Repetición 5: 71.37%
Promedio: 71.37%
Desvío estandar: 1.51%
Intervalo de confianza del 95%: [69.49, 73.25]%
-----------------------------------------------------------------
llama3.2:3b
Precisión Repetición 1: 71.37%
Precisión Repetición 2: 76.92%
Precisión Repetición 3: 78.21%
Precisión Repetición 4: 73.08%
Precisión Repetición 5: 77.78%
Promedio: 75.47%
